# VISCERA · Optional atlas text-to-spec adapter

This separate research workflow trains **LoRA adapters** on paired requests and
complete, bounded atlas specifications. It uses the immutable Apache-2.0
[Qwen2.5-0.5B-Instruct revision](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/tree/c89bee90d9f811437d9735454613c35b4a3c4dc8).
The base weights stay frozen. This is **not** a text-to-mesh foundation model.

The operational studio already uses a conservative deterministic compiler.
Learned outputs remain untrusted proposals: strict JSON, complete schema,
compiler agreement and geometry QA are required before realization. A model that
fails to exactly preserve state or requested controls will be rejected.

The local delivery exercised actual tiny-Qwen LoRA optimization and exact resume
on CPU. **Training the full pretrained Qwen adapter and GPU Colab execution are
NOT_RUN locally.** This notebook supplies that optional next experiment. It never
automatically replaces the studio compiler or promotes an adapter.

## 1 · Load the bundle
Upload `viscera-atlas-colab.zip`, or run from an extracted `viscera` folder.


In [ ]:
from pathlib import Path
import json, os, stat, sys, zipfile

candidates = [Path.cwd(), Path.cwd() / "viscera", Path("/content/viscera")]
ROOT = next((p for p in candidates if (p / "pipelines/train/viscera_train/atlas_shape.py").is_file()), None)
if ROOT is None:
    from google.colab import files
    uploaded = files.upload()
    archives = [Path(name) for name in uploaded if name.endswith(".zip")]
    if len(archives) != 1:
        raise ValueError("Upload one VISCERA atlas bundle ZIP")
    target = Path("/content").resolve()
    with zipfile.ZipFile(archives[0]) as archive:
        if sum(member.file_size for member in archive.infolist()) > 2_000_000_000:
            raise ValueError("Unexpectedly large bundle")
        for member in archive.infolist():
            path = (target / member.filename).resolve()
            if not path.is_relative_to(target) or stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError("Unsafe archive member")
        archive.extractall(target)
    ROOT = target / "viscera"
if not (ROOT / "pipelines/train/viscera_train/atlas_shape.py").is_file():
    raise FileNotFoundError("Bundle lacks atlas trainer")
os.chdir(ROOT)
sys.path[:0] = [str(ROOT), str(ROOT / "py")]
print("Project:", ROOT)


## 2 · Install pinned dependencies
Use Python 3.11 or 3.12. Select a GPU runtime for the pretrained-model experiment.
The exact lock pins Transformers 4.57.1, PEFT 0.17.1 and Accelerate 1.10.1.
The download includes approximately 1 GB of base weights from the pinned revision.


In [ ]:
import subprocess
if not ((3, 11) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError("Use a Python 3.11 or 3.12 runtime")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "--require-hashes",
                    "-r", str(ROOT / "notebooks/requirements-atlas-language-colab.txt")], check=True)
import numpy as np
import torch
print("Python", sys.version.split()[0], "NumPy", np.__version__, "PyTorch", torch.__version__)


In [ ]:
from pipelines.train.viscera_train import atlas_language as language
from pipelines.train.viscera_train import atlas_shape as storage
from viscera_core.atlas_dataset import verify_release

PARENT = ROOT / "datasets/atlas/cohort-v2"
DATA = ROOT / "datasets/atlas/language-v1"
OUT = ROOT / "artifacts/atlas/language-colab-run-v1"
verify_release(PARENT)
release = language.verify_contract(DATA, PARENT)
train = language.read_split(DATA, "train")
val = language.read_split(DATA, "val")
language.audit_groups(train, val)
print("Parent release:", json.loads((PARENT / "release.json").read_text())["release_id"])
print("Language rows:", {"train": len(train), "val": len(val)})
print("Audit examples have not been parsed.")
print("Context:", train[0]["context_spec"])
print("Request:", train[0]["text"])
print("Target:", language.target(train[0]))


##  · Train the optional adapter
The train/validation/audit partitions inherit geometry parent groups. Rendering
or paraphrasing a parent cannot move its descendants into another partition.
These are synthetic compiler-generated requests, not independent human language.

The trainer masks prompt tokens and supervises the **entire** assistant answer.
It rejects truncation, validates every row, uses safe checkpoint weights, and
binds resume to exact sources, data, device, dependency versions and configuration.
Baseline and adapter are evaluated on the same organ/status-balanced validation
examples. Invalid proposals are counted and never applied to geometry.

Leave `RUN_TRAINING = False` to inspect the data and compiler without downloading
weights. Change it to `True` to start the optional experiment on a GPU. A 200-step
run is an initial research experiment; it is not a demonstrated quality target.


In [ ]:
RUN_TRAINING = False
CONFIG = language.LanguageConfig(steps=200, checkpoint_every=25, validation_limit=24,
                                 maximum_tokens=2048, device="cuda")
if RUN_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError("Select a GPU runtime for this optional pretrained-model experiment")
    report = language.train_lora(DATA, PARENT, OUT, CONFIG,
                                 resume=(OUT / "checkpoint.pt").is_file())
    print(json.dumps({k: v for k, v in report.items() if k != "losses"}, indent=2))
    print("Frozen-base validation:")
    print((OUT / "frozen_base_validation.json").read_text())
else:
    print("Pretrained Qwen adapter training: NOT_RUN. Dataset/compiler inspection remains available.")


## 4 · Inspect the strict geometry boundary
This cell exercises the real paired target through the same guard used for model
proposals. The guard checks every state field against the deterministic compiler
before high-resolution realization. It is not evidence that a language model
produced this answer. Only the measured high-resolution atlas decoder determines
geometry; text output can never inject code or arbitrary topology.


In [ ]:
example = next(row for row in val if row["expected_status"] == "accepted")
proposal = language.canonical(language.target(example))
checked = language.guard_proposal(proposal, PARENT / "templates", example["text"], example["context_spec"])
sample = language.realize_checked(proposal, PARENT / "templates", example["text"], example["context_spec"])
print("Paired compiler target:", example["text"])
print("Checked status:", checked["status"], "Surface QA:", sample.qa)
print("Measurements:", sample.measurements)


## 5 · Inspect learned proposals without automatic promotion
The loader verifies every adapter file and its source/dataset provenance before
loading safe weights. Proposed text is checked; disagreement is reported as a
rejection. No checkpoint is automatically installed into the studio.


In [ ]:
if RUN_TRAINING:
    model, tokenizer = language.load_adapter(DATA, PARENT, OUT, device="cuda")
    proposals = language.predictions(model, tokenizer, [example], "cuda")
    print("Learned proposal:", proposals[0])
    try:
        print("Guard result:", language.guard_proposal(proposals[0], PARENT / "templates",
                                                       example["text"], example["context_spec"]))
    except (ValueError, TypeError, KeyError) as error:
        print("REJECTED; no geometry was generated:", str(error))
else:
    print("Learned inference: NOT_RUN")


## 6 · Preserve evidence
The training path never reads audit examples. Freeze the chosen adapter and
validation decision before any future audit evaluation. This notebook intentionally
leaves audit closed. Report schema validity, supported/unsupported decisions,
compiler semantic agreement and per-organ results; do not infer language or
clinical generalization from synthetic policy replication.

The geometry source licenses and lineage limitations still apply. Mechanical
material calibration, pathological variation, internal anatomy and scanned-tissue
textures are outside this release. See `docs/ATLAS_LANGUAGE_TRAINING.md`.


In [ ]:
if RUN_TRAINING:
    import shutil
    archive = shutil.make_archive(str(OUT), "zip", OUT)
    print("Saved unpromoted adapter and metrics:", archive)
    if "google.colab" in sys.modules:
        from google.colab import files
        files.download(archive)
